# Análisis de Geometría — Termalización AmBe
Cómo afectan las dimensiones de la parafina (Ancho × Alto × Espesor) a la termalización y detección.
Un panel por cada distancia fuente–parafina.

In [11]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import uproot
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams.update({'figure.dpi': 130, 'font.size': 11,
                     'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})

BUILD_DIR = os.path.abspath('../build')
CSV_PATH  = os.path.join(BUILD_DIR, 'resultados_ambe.csv')

df = pd.read_csv(CSV_PATH)
df['Efic_deteccion_%'] = df['Neutrones'] / df['BeamOn'] * 100
df['Area_frontal_cm2'] = df['Ancho_cm'] * df['Alto_cm']
df['Cara_cuadrada']    = df['Ancho_cm'] == df['Alto_cm']

BEAM_ON  = int(df['BeamOn'].iloc[0])
ACTIVITY = float(df['Actividad_Ci'].iloc[0])
distancias = sorted(df['Distancia_cm'].unique())

print(f'Configuraciones: {len(df)}')
print(f'Distancias (cm): {distancias}')
print(f'Espesores  (cm): {sorted(df.Espesor_cm.unique())}')
print(f'Anchos     (cm): {sorted(df.Ancho_cm.unique())}')
print(f'Altos      (cm): {sorted(df.Alto_cm.unique())}')
print(f'Caras cuadradas: {df.Cara_cuadrada.sum()} / {len(df)}')
df.head(3)

Configuraciones: 36
Distancias (cm): [np.float64(5.0), np.float64(10.0), np.float64(20.0)]
Espesores  (cm): [np.float64(10.0), np.float64(20.0), np.float64(30.0)]
Anchos     (cm): [np.float64(10.0), np.float64(20.0)]
Altos      (cm): [np.float64(10.0), np.float64(20.0)]
Caras cuadradas: 18 / 36


,Ancho_cm,Alto_cm,Espesor_cm,Distancia_cm,BeamOn,Actividad_Ci,Duracion_s,Total_detectados,Neutrones,Gammas,Electrones,Otros,N_termicos,N_epitermicos,N_rapidos,Frac_termica_%,Tasa_n_det_ps,Efic_deteccion_%,Area_frontal_cm2,Cara_cuadrada
0,10.0,10.0,10.0,5.0,50000,2.98,4.6,103,45,55,3,0,1,22,22,2.22,5900.40,0.090,100.0,True
1,10.0,10.0,10.0,10.0,50000,2.98,3.0,53,26,25,2,0,0,8,18,0.00,3409.12,0.052,100.0,True
2,10.0,10.0,10.0,20.0,50000,2.98,2.2,23,9,14,0,0,0,4,5,0.00,1180.08,0.018,100.0,True


## 1. Composición de partículas por distancia
Top-15 geometrías con más detecciones para cada distancia.

In [ ]:
colores_p = {'Neutrones': '#1565C0', 'Gammas': '#E65100', 'Electrones': '#2E7D32'}

fig, axes = plt.subplots(1, len(distancias), figsize=(7*len(distancias), 5), sharey=False)
if len(distancias) == 1:
    axes = [axes]

for ax, dist in zip(axes, distancias):
    sub = df[df.Distancia_cm == dist].nlargest(15, 'Total_detectados').copy()
    sub = sub.sort_values('Total_detectados', ascending=True)
    labels = [f"{int(r.Ancho_cm)}×{int(r.Alto_cm)}×{int(r.Espesor_cm)}" for _, r in sub.iterrows()]
    y = np.arange(len(labels))
    h = 0.6
    ax.barh(y, sub['Neutrones'],  h, label='Neutrones',  color='#1565C0')
    ax.barh(y, sub['Gammas'],     h, left=sub['Neutrones'], label='Gammas', color='#E65100')
    ax.barh(y, sub['Electrones'], h, left=sub['Neutrones']+sub['Gammas'], label='e-', color='#2E7D32')
    ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('Partículas detectadas')
    ax.set_title(f'Distancia = {int(dist)} cm')
    ax.legend(fontsize=8)

plt.suptitle('Composición de partículas (Ancho×Alto×Espesor en cm)', fontsize=12)
plt.tight_layout(); plt.show()

: 

: 

: 

: 

## 2. Mapa de calor — Fracción térmica vs Espesor y lado (caras cuadradas)
Filtrado a caras cuadradas (Ancho = Alto) para facilitar la visualización. Un mapa por distancia.

In [ ]:
sq = df[df['Cara_cuadrada']].copy()
vmax = sq['Frac_termica_%'].max() if sq['Frac_termica_%'].max() > 0 else 1

fig, axes = plt.subplots(1, len(distancias), figsize=(6*len(distancias), 5))
if len(distancias) == 1:
    axes = [axes]

for ax, dist in zip(axes, distancias):
    sub = sq[sq.Distancia_cm == dist]
    pivot = sub.pivot_table(values='Frac_termica_%',
                            index='Espesor_cm', columns='Ancho_cm')
    pivot.index   = [f'{v:.0f}' for v in pivot.index]
    pivot.columns = [f'{v:.0f}' for v in pivot.columns]
    sns.heatmap(pivot, ax=ax, annot=True, fmt='.1f', cmap='YlOrRd',
                linewidths=0.4, vmin=0, vmax=vmax,
                cbar_kws={'label': 'Térmica (%)'})
    ax.set_title(f'Distancia = {int(dist)} cm')
    ax.set_xlabel('Lado de la cara (cm)')
    ax.set_ylabel('Espesor (cm)')

plt.suptitle('Fracción de termalización (%) — caras cuadradas (Ancho = Alto)', fontsize=12)
plt.tight_layout(); plt.show()

: 

: 

: 

: 

## 3. Efecto del área frontal (cara no cuadrada)
Todos los datos. X = área frontal (Ancho × Alto), Y = Espesor. Color = fracción térmica.
El área captura el efecto combinado de Ancho y Alto sin asumir que son iguales.

In [ ]:
fig, axes = plt.subplots(1, len(distancias), figsize=(6*len(distancias), 5))
if len(distancias) == 1:
    axes = [axes]

for ax, dist in zip(axes, distancias):
    sub = df[df.Distancia_cm == dist]
    sc = ax.scatter(sub['Area_frontal_cm2'], sub['Espesor_cm'],
                    c=sub['Frac_termica_%'], cmap='plasma',
                    s=60, edgecolors='grey', linewidths=0.3,
                    vmin=0, vmax=df['Frac_termica_%'].max() or 1)
    plt.colorbar(sc, ax=ax, label='Térmica (%)')
    ax.set_xlabel('Área frontal (cm²)  [Ancho × Alto]')
    ax.set_ylabel('Espesor (cm)')
    ax.set_title(f'Distancia = {int(dist)} cm')

plt.suptitle('Fracción de termalización — área frontal no cuadrada', fontsize=12)
plt.tight_layout(); plt.show()

# También: eficiencia de detección
fig, axes = plt.subplots(1, len(distancias), figsize=(6*len(distancias), 5))
if len(distancias) == 1:
    axes = [axes]

for ax, dist in zip(axes, distancias):
    sub = df[df.Distancia_cm == dist]
    sc = ax.scatter(sub['Area_frontal_cm2'], sub['Espesor_cm'],
                    c=sub['Efic_deteccion_%'], cmap='viridis',
                    s=60, edgecolors='grey', linewidths=0.3)
    plt.colorbar(sc, ax=ax, label='Efic. detección (%)')
    ax.set_xlabel('Área frontal (cm²)')
    ax.set_ylabel('Espesor (cm)')
    ax.set_title(f'Distancia = {int(dist)} cm')

plt.suptitle('Eficiencia de detección de neutrones — área frontal no cuadrada', fontsize=12)
plt.tight_layout(); plt.show()

: 

: 

: 

: 

## 4. Superficie 3D — Fracción térmica
Eje X = Área frontal, Eje Y = Espesor, Eje Z = fracción de termalización. Un panel por distancia.

In [ ]:
fig = plt.figure(figsize=(7*len(distancias), 6))

for idx, dist in enumerate(distancias, 1):
    ax = fig.add_subplot(1, len(distancias), idx, projection='3d')
    sub = df[df.Distancia_cm == dist]
    sc = ax.scatter(sub['Area_frontal_cm2'], sub['Espesor_cm'],
                    sub['Frac_termica_%'],
                    c=sub['Frac_termica_%'], cmap='plasma',
                    s=40, edgecolors='k', linewidths=0.3)
    ax.set_xlabel('Área frontal (cm²)', labelpad=6)
    ax.set_ylabel('Espesor (cm)',        labelpad=6)
    ax.set_zlabel('Térmica (%)',         labelpad=6)
    ax.set_title(f'Distancia = {int(dist)} cm')
    fig.colorbar(sc, ax=ax, shrink=0.5, label='Térmica (%)')

plt.suptitle('Fracción de termalización — vista 3D', fontsize=13)
plt.tight_layout(); plt.show()

: 

: 

: 

: 

## 5. Distribución de energías por tipo de partícula
Archivos ROOT seleccionados: distintos espesores para cada distancia disponible.

In [ ]:
def root_path(ancho, alto, espesor, dist):
    return os.path.join(BUILD_DIR,
        f'AmBe_{int(ancho)}x{int(alto)}x{int(espesor)}cm_d{int(dist)}cm.root')

def cargar_energia(ancho, alto, espesor, dist):
    p = root_path(ancho, alto, espesor, dist)
    if not os.path.exists(p):
        return None, None
    with uproot.open(p) as f:
        arr = f['PhaseSpace'].arrays(['Particle','KinE_eV'], library='np')
    return arr['Particle'], arr['KinE_eV']

part_cfg = {
    'neutron': ('#1565C0', 'Neutrón'),
    'gamma':   ('#E65100', 'Gamma'),
    'e-':      ('#2E7D32', 'e⁻'),
}
bins = np.logspace(-3, 7, 80)

# Para cada distancia, escoge hasta 4 espesores representativos con caras cuadradas
for dist in distancias:
    sq_d = df[(df.Distancia_cm == dist) & df.Cara_cuadrada]
    if sq_d.empty:
        continue
    # Elige geometrias con mas detecciones
    tops = sq_d.nlargest(4, 'Total_detectados')[['Ancho_cm','Alto_cm','Espesor_cm']].values

    fig, axes = plt.subplots(1, len(tops), figsize=(5*len(tops), 4), sharey=False)
    if len(tops) == 1:
        axes = [axes]

    for ax, (ancho, alto, esp) in zip(axes, tops):
        part, ene = cargar_energia(ancho, alto, esp, dist)
        if part is None:
            ax.text(0.5, 0.5, 'No encontrado', ha='center', va='center',
                    transform=ax.transAxes, color='red')
            continue
        for pname, (col, lbl) in part_cfg.items():
            mask = part == pname
            if not mask.any():
                continue
            ax.hist(ene[mask], bins=bins, histtype='stepfilled',
                    alpha=0.5, color=col, label=f'{lbl} ({mask.sum()})')
            ax.hist(ene[mask], bins=bins, histtype='step', color=col, lw=1.5)
        ax.axvline(0.025, color='k', ls='--', lw=1.2, label='E_térmica')
        ax.set_xscale('log')
        ax.set_xlabel('Energía (eV)')
        ax.set_ylabel('Cuentas')
        ax.set_title(f'{int(ancho)}×{int(alto)}×{int(esp)} cm')
        ax.legend(fontsize=7)

    plt.suptitle(f'Distribución de energías — Distancia {int(dist)} cm', fontsize=12)
    plt.tight_layout(); plt.show()

: 

: 

: 

: 